In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from xgboost import XGBClassifier
import pickle

In [2]:
malaria = pd.read_csv(r'datasets\Malaria_Dataset.csv')

In [3]:
malaria.head()

,IP_Number,Age,Sex,Residence_Area,DOA,Discharge_Date,Fever,Headache,Abdominal_Pain,General_Body_Malaise,...,Vomiting,Confusion,Backache,Chest_Pain,Coughing,Joint_Pain,Primary_Code,Diagnosis_Type,Target,Risk_Score
0,14xxxx31,52,Female,Mangalore,31-10-2015 20:42,05-11-2015 05:16,0,0,0,1,...,0,0,1,0,0,0,B50.9,Mixed Malaria Infection,0,3
1,28xxxx34,75,Female,Shimoga,03-02-2015 23:28,13-02-2015 19:27,1,0,1,1,...,0,1,0,1,1,1,B50.9,Mixed Malaria Infection,1,11
2,96xxxx43,30,Female,Mangalore,15-11-2019 12:31,19-11-2019 14:31,1,1,1,1,...,0,1,1,1,0,1,B50.9,Mixed Malaria Infection,1,13
3,49xxxx87,89,Female,Mangalore,17-05-2017 17:50,23-05-2017 13:22,0,0,0,0,...,1,1,1,1,0,1,B54,Plasmodium vivax Malaria without complication,0,5
4,48xxxx10,62,Male,Shimoga,26-06-2015 15:29,27-06-2015 23:35,0,1,0,1,...,1,1,0,0,0,0,B51.0,Plasmodium falciparum Malaria without complica...,1,8


In [4]:
malaria.columns

Index(['IP_Number', 'Age', 'Sex', 'Residence_Area', 'DOA', 'Discharge_Date',
       'Fever', 'Headache', 'Abdominal_Pain', 'General_Body_Malaise',
       'Dizziness', 'Vomiting', 'Confusion', 'Backache', 'Chest_Pain',
       'Coughing', 'Joint_Pain', 'Primary_Code', 'Diagnosis_Type', 'Target',
       'Risk_Score'],
      dtype='object')

## Feature Engineering

In [5]:
malaria = malaria.drop(columns=['IP_Number', 'Diagnosis_Type', 'DOA', 'Discharge_Date', 'Risk_Score', 'Primary_Code', 'Residence_Area'])

In [6]:
symptom_cols = [
    'Fever','Headache','Abdominal_Pain','General_Body_Malaise',
    'Dizziness','Vomiting','Confusion','Backache',
    'Chest_Pain','Coughing','Joint_Pain'
]

malaria['symptom_count'] = malaria[symptom_cols].sum(axis=1)

In [7]:
malaria['fever_headache'] = malaria['Fever'] * malaria['Headache']
malaria['fever_vomiting'] = malaria['Fever'] * malaria['Vomiting']

In [8]:
malaria['age_group'] = pd.cut(
    malaria['Age'],
    bins=[0,5,18,50,100],
    labels=['child','teen','adult','older']
)

## Feature Encoding

In [9]:
malaria.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1622 entries, 0 to 1621
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   Age                   1622 non-null   int64   
 1   Sex                   1622 non-null   object  
 2   Fever                 1622 non-null   int64   
 3   Headache              1622 non-null   int64   
 4   Abdominal_Pain        1622 non-null   int64   
 5   General_Body_Malaise  1622 non-null   int64   
 6   Dizziness             1622 non-null   int64   
 7   Vomiting              1622 non-null   int64   
 8   Confusion             1622 non-null   int64   
 9   Backache              1622 non-null   int64   
 10  Chest_Pain            1622 non-null   int64   
 11  Coughing              1622 non-null   int64   
 12  Joint_Pain            1622 non-null   int64   
 13  Target                1622 non-null   int64   
 14  symptom_count         1622 non-null   int64   
 15  feve

In [10]:
categorical_cols = malaria.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols

['Sex', 'age_group']

In [11]:
# Fix: use a separate LabelEncoder per column and store each one
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    label_encoders[col] = le
    malaria[col] = le.fit_transform(malaria[col])

## Train-Test split

In [12]:
x = malaria.drop(columns=['Target'])
y = malaria['Target']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [14]:
# Compute class imbalance ratio for XGBoost (Fix 4)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight_ratio = neg_count / pos_count
print(f"Negative samples: {neg_count}, Positive samples: {pos_count}")
print(f"scale_pos_weight ratio: {scale_pos_weight_ratio:.4f}")

Negative samples: 363, Positive samples: 934
scale_pos_weight ratio: 0.3887


In [15]:
# Fix: Run cross-validation INSIDE pipelines to prevent data leakage.
# The scaler is fitted only on each training fold, never on validation data.
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=32))
])

xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(scale_pos_weight=scale_pos_weight_ratio, random_state=32))
])

# Cross-validate on raw (unscaled) X_train — scaling happens inside each fold
rf_cv_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='roc_auc')
xgb_cv_scores = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='roc_auc')

In [16]:
print(f"Random Forest CV AUC-ROC: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std():.4f})")
print(f"XGBoost CV AUC-ROC:       {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std():.4f})")

Random Forest CV AUC-ROC: 0.9858 (+/- 0.0051)
XGBoost CV AUC-ROC:       0.9970 (+/- 0.0035)


## Feature scaling

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test) 

## Train Models
- Random Forest
- XGBoost

In [18]:
# Fix: XGBoost now uses scale_pos_weight to handle class imbalance,
# matching the balanced treatment given to Random Forest.
random_forest_model = RandomForestClassifier(class_weight='balanced', random_state=32)
xgboost_model = XGBClassifier(scale_pos_weight=scale_pos_weight_ratio, random_state=32)

In [19]:
# Cross-validation was already performed above using leakage-free pipelines.
# See rf_cv_scores and xgb_cv_scores for results.

In [20]:
# CV scores summary (already printed above)
print(f"Random Forest CV AUC-ROC: {rf_cv_scores.mean():.4f}")
print(f"XGBoost CV AUC-ROC:       {xgb_cv_scores.mean():.4f}")

Random Forest CV AUC-ROC: 0.9858
XGBoost CV AUC-ROC:       0.9970


In [21]:
random_forest_model.fit(X_train, y_train)
xgboost_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [22]:
random_forest_predictions = random_forest_model.predict(X_test)
xgboost_predictions = xgboost_model.predict(X_test)

## Model evaluation

In [23]:
print(f"Random Forest Accuracy: {accuracy_score(y_test, random_forest_predictions):.4f}")
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgboost_predictions):.4f}")

Random Forest Accuracy: 0.9415
XGBoost Accuracy: 0.9846


In [24]:
print(classification_report(y_test, random_forest_predictions, target_names=["NEGATIVE", "POSITIVE"]))
print("____________________________________________________________________")
print(classification_report(y_test, xgboost_predictions, target_names=["NEGATIVE", "POSITIVE"]))

              precision    recall  f1-score   support

    NEGATIVE       0.93      0.86      0.89        92
    POSITIVE       0.95      0.97      0.96       233

    accuracy                           0.94       325
   macro avg       0.94      0.92      0.93       325
weighted avg       0.94      0.94      0.94       325

____________________________________________________________________
              precision    recall  f1-score   support

    NEGATIVE       0.98      0.97      0.97        92
    POSITIVE       0.99      0.99      0.99       233

    accuracy                           0.98       325
   macro avg       0.98      0.98      0.98       325
weighted avg       0.98      0.98      0.98       325



In [25]:
# Fix: Use predicted probabilities (not hard labels) for a proper AUC-ROC curve.
random_forest_auc_score = roc_auc_score(y_test, random_forest_model.predict_proba(X_test)[:, 1])
xgboost_auc_score = roc_auc_score(y_test, xgboost_model.predict_proba(X_test)[:, 1])

In [26]:
print(f"Random Forest AUC-ROC: {random_forest_auc_score:.4f}")
print(f"XGBoost AUC-ROC: {xgboost_auc_score:.4f}")

Random Forest AUC-ROC: 0.9939
XGBoost AUC-ROC: 0.9984


## Feature importance

In [28]:
importance = xgboost_model.feature_importances_
features = x.columns

feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print(feature_importance)

                 Feature  Importance
13         symptom_count    0.262900
14        fever_headache    0.157068
15        fever_vomiting    0.092723
2                  Fever    0.073842
5   General_Body_Malaise    0.065054
10            Chest_Pain    0.040209
11              Coughing    0.037720
6              Dizziness    0.036202
0                    Age    0.035887
3               Headache    0.034359
7               Vomiting    0.032494
4         Abdominal_Pain    0.030848
9               Backache    0.026056
16             age_group    0.023186
12            Joint_Pain    0.020985
8              Confusion    0.016758
1                    Sex    0.013708


## Export model, encoder and scaler

In [31]:
with open('models\model.pkl', 'wb') as file:
    pickle.dump(xgboost_model,file)

In [32]:
with open('models\scaler.pkl', 'wb') as file:
    pickle.dump(scaler,file)

In [33]:
with open('models\encoder.pkl', 'wb') as file:
    pickle.dump(le,file)